In [1]:
# Link zum Code: https://github.com/JayPatwardhan/ResNet-PyTorch/blob/master    /ResNet/ResNet.py
# Originalpaper: https://arxiv.org/pdf/1512.03385-

In [2]:
import time
import cv2
import random
import glob
import math

from tqdm import tqdm

from PIL import Image
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from Models.my_resnet import ResNet50 as resnet50

In [3]:
import os

os.makedirs("./Models/Save_Weights", exist_ok=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
from torch.nn.utils import clip_grad_norm_
from torch.amp import autocast, GradScaler

import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, LambdaLR

import torchvision
from torchvision import transforms
from torchsummary import summary


print(torchvision.__file__)

# Set environment variables for PyTorch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Specify which GPU to use, if needed

# PyTorch does not have direct equivalents for some TensorFlow environment settings,
# but you can manage GPU memory growth and logging through PyTorch"s API.

# Disable debug information (PyTorch does not have a direct equivalent, but you can manage logging)
# PyTorch logging can be managed through Python"s logging module or by setting verbosity levels.

torch.set_float32_matmul_precision('high')

# Print PyTorch version
print(torch.__version__)

# Print number of available GPUs
print("Num GPUs Available: ", torch.cuda.device_count())

# List local devices (PyTorch does not have a direct equivalent, but you can check CUDA devices)
for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

/home/cgd/miniconda3/envs/vit/lib/python3.12/site-packages/torchvision/__init__.py
2.6.0+cu126
Num GPUs Available:  1
Device 0: NVIDIA GeForce RTX 4090


In [4]:
df = pd.read_csv('./Dataset/train.csv')
print(len(df['writer_id'].unique()))   
df.head()

44


,image_id,image_path,writer_id,pen_id
0,4,images/00004.png,W27,8
1,5,images/00005.png,W17,1
2,7,images/00007.png,W01,8
3,8,images/00008.png,W17,5
4,9,images/00009.png,W24,4


In [5]:
df_test = pd.read_csv('./Dataset/test.csv')
df_test.head()

,image_id,image_path
0,v2_8eb750cb7bac5c42036af72b8253976b,images/v2_8eb750cb7bac5c42036af72b8253976b.png
1,v2_04e19b0acea03fe2ae474ce8a4c6b705,images/v2_04e19b0acea03fe2ae474ce8a4c6b705.png
2,v2_6b3400d5252124adcc9859cbc78c5d8a,images/v2_6b3400d5252124adcc9859cbc78c5d8a.png
3,v2_79025cb2ef36af3dc15c91056fa225dc,images/v2_79025cb2ef36af3dc15c91056fa225dc.png
4,v2_3ed2f62c32c69ec191b7e5b86433cb87,images/v2_3ed2f62c32c69ec191b7e5b86433cb87.png


In [ ]:
# ── Config ────────────────────────────────────────────────
NUM_CLASSES          = len(df['writer_id'].unique())
BATCH_SIZE           = 64
EPOCHS               = 50
LR                   = 1e-4
IMG_SIZE             = 224
CONFIDENCE_THRESHOLD = 0.5
DEVICE               = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("./Models/Save_Weights", exist_ok=True)
os.makedirs("./Submissions",         exist_ok=True)

In [7]:
# ── 1. Daten ──────────────────────────────────────────────
df = pd.read_csv('./Dataset/train.csv')

le = LabelEncoder()
df['label'] = le.fit_transform(df['writer_id'])

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

In [8]:
# ── 2. Dataset ────────────────────────────────────────────
class WriterDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = Image.open(f"./Dataset/{row['image_path']}").convert("RGB")
        label = torch.tensor(row['label'], dtype=torch.long)
        return self.transform(image), label

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(WriterDataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(WriterDataset(val_df, val_transform),
                          batch_size=BATCH_SIZE, shuffle=False)


In [9]:
# ── 3. Modell ─────────────────────────────────────────────
model = resnet50(num_classes=NUM_CLASSES)   # vortrainiert

# Nur den letzten Layer anpassen (deine Klassen statt 1000)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

In [10]:
# ── 4. Loss, Optimizer, Scheduler, EarlyStopping ─────────
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True
)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, path="best_model.pth"):
        self.patience  = patience
        self.min_delta = min_delta
        self.path      = path
        self.best_loss = float('inf')
        self.counter   = 0
        self.stopped   = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
            print(f"  ✔ Neues bestes Modell gespeichert (val_loss={val_loss:.4f})")
        else:
            self.counter += 1
            print(f"  EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.stopped = True

early_stopping = EarlyStopping(patience=5, min_delta=0.001,
                                path="./Models/Save_Weights/best_model.pth")
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

/home/cgd/miniconda3/envs/vit/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
# ── 5. Trainingsloop ──────────────────────────────────────
for epoch in range(EPOCHS):

    model.train()
    train_loss, train_correct = 0.0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()

    model.eval()
    val_loss, val_correct = 0.0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs        = model(images)
            loss           = criterion(outputs, labels)
            val_loss      += loss.item() * images.size(0)
            probs          = torch.softmax(outputs, dim=1)
            confidence, predicted = probs.max(dim=1)
            mask           = confidence >= CONFIDENCE_THRESHOLD
            val_correct   += (predicted[mask] == labels[mask]).sum().item()

    t_loss = train_loss / len(train_df)
    t_acc  = train_correct / len(train_df)
    v_loss = val_loss / len(val_df)
    v_acc  = val_correct / len(val_df)

    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {t_loss:.4f}  Acc: {t_acc:.4f} | "
        f"Val Loss:   {v_loss:.4f}  Acc: {v_acc:.4f}"
    )

    scheduler.step(v_loss)
    early_stopping.step(v_loss, model)

    if early_stopping.stopped:
        print(f"\nEarlyStopping nach Epoch {epoch+1}.")
        break

# ── 6. Bestes Modell laden & final speichern ──────────────
model.load_state_dict(torch.load("./Models/Save_Weights/best_model.pth"))
torch.save(model.state_dict(), "./Models/Save_Weights/resnet50_final.pth")
print("Training abgeschlossen. Modell gespeichert.")


Epoch 01/20 | Train Loss: 2.8303  Acc: 0.2007 | Val Loss:   5.3965  Acc: 0.0549
  ✔ Neues bestes Modell gespeichert (val_loss=5.3965)
Epoch 02/20 | Train Loss: 2.0390  Acc: 0.3939 | Val Loss:   3.7760  Acc: 0.1046
  ✔ Neues bestes Modell gespeichert (val_loss=3.7760)


In [ ]:
# ── 7. Inference & Submission ─────────────────────────────
class TestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = Image.open(f"./Dataset/{row['image_path']}").convert("RGB")
        return self.transform(image), row['image_id']

test_df     = pd.read_csv('./Dataset/test.csv')
test_loader = DataLoader(TestDataset(test_df, val_transform),
                         batch_size=BATCH_SIZE, shuffle=False)

model.eval()
all_ids, all_preds = [], []

with torch.no_grad():
    for images, image_ids in test_loader:
        images      = images.to(DEVICE)
        probs       = torch.softmax(model(images), dim=1)
        confidence, predicted = probs.max(dim=1)

        for conf, pred, img_id in zip(confidence, predicted, image_ids):
            if conf.item() < CONFIDENCE_THRESHOLD:
                all_preds.append(-1)
            else:
                all_preds.append(le.inverse_transform([pred.item()])[0])
            all_ids.append(img_id.item())

submission = pd.DataFrame({"image_id": all_ids, "writer_id": all_preds})
submission.to_csv("./Submissions/submission.csv", index=False)
print(submission.head())
print(f"Unbekannte Writer (-1): {(submission['writer_id'] == -1).sum()}")
print(f"Bekannte Writer:        {(submission['writer_id'] != -1).sum()}")